**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/02-finding-corners.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/finding-corners) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6B · Finding Chessboard Corners

_Lecture 6, Section 5: calibrating a camera, finding corners._

The first step of calibration: find the **image points** (detected corners, pixels) and pair them with the
**object points** (where those corners really are on the flat board). We use the Udacity GoPro calibration set:
36 pictures of the same chessboard at different angles and distances.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('gopro')
print("data ready")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C

images = sorted(glob.glob("data/calibration_wide/GOPR*.jpg"))
print(len(images), "calibration images")
fig, ax = plt.subplots(2, 4, figsize=(18, 6.5))
for a, f in zip(ax.ravel(), images[::5]):
    a.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); a.set_title(f.split("/")[-1].split("\\")[-1]); a.axis("off")
plt.show()

## 1. Quiz: count the corners

Look at the images above. Count the **inside corners** in a row (`nx`) and in a column (`ny`): only points where
**two black and two white** squares meet, not the outside corners.

In [ ]:
nx = 0   # TODO: corners per row
ny = 0   # TODO: corners per column

In [ ]:
C.check_pattern_size(nx, ny)

## 2. `findChessboardCorners` and `drawChessboardCorners`

```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # cv2.imread gives BGR!  (mpimg.imread gives RGB)
ret, corners = cv2.findChessboardCorners(gray, (nx, ny), None)
img = cv2.drawChessboardCorners(img, (nx, ny), corners, ret)
```

In [ ]:
img = cv2.imread(images[0])
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, (nx, ny), None)
print("found:", ret, "  corners shape:", None if corners is None else corners.shape)
if ret:
    vis = cv2.drawChessboardCorners(img.copy(), (nx, ny), corners, ret)
    plt.figure(figsize=(12, 9)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

> **OpenCV 5 note.** `findChessboardCorners` now returns corners of shape **(N, 2)**; OpenCV 4.x returned **(N, 1, 2)**.
> `corners.reshape(-1, 2)` gives `(N, 2)` in both, and `corners.reshape(-1, 1, 2)` gives the old layout that some functions expect.

The **first** corner (red row start) becomes object point (0, 0, 0). Corners are ordered row by row, **x fastest**.

## 3. Object points

For an `nx × ny` board the object points are `(0,0,0), (1,0,0), …, (nx-1, ny-1, 0)`, with **z = 0** because the board is flat.
Build `objp` with shape `(nx*ny, 3)` and dtype `float32` using `np.zeros` and `np.mgrid`.

In [ ]:
# TODO
objp = np.zeros((nx * ny, 3), np.float32)

print(objp[:10])

In [ ]:
C.check_object_points(objp, nx, ny)

<details><summary><b>Show a solution</b></summary>

```python
objp = np.zeros((nx * ny, 3), np.float32)
objp[:, :2] = np.mgrid[0:nx, 0:ny].T.reshape(-1, 2)
```
`np.mgrid[0:nx, 0:ny]` has shape (2, nx, ny); `.T` makes it (ny, nx, 2), so x changes fastest after the reshape,
exactly like the detected corners.
</details>

## 4. Every image: collect object points and image points

In [ ]:
objpoints, imgpoints, used = [], [], []
for fname in images:
    gray = cv2.imread(fname, cv2.IMREAD_GRAYSCALE)
    ret, corners = cv2.findChessboardCorners(gray, (nx, ny), None)
    if ret:
        objpoints.append(objp)
        imgpoints.append(corners.reshape(-1, 1, 2))
        used.append(fname)
print(f"board found in {len(imgpoints)} / {len(images)} images")

In [ ]:
C.check_found(len(imgpoints), len(images))

## 5. Which image failed, and why?

In [ ]:
failed = [f for f in images if f not in used]
fig, ax = plt.subplots(1, len(failed), figsize=(8 * len(failed), 6), squeeze=False)
for a, f in zip(ax[0], failed):
    a.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); a.set_title("NOT found: " + f[-12:]); a.axis("off")
plt.show()

## 6. Sub-pixel refinement and the `SB` detector

`cornerSubPix` moves each corner to the exact saddle point, typically to ~0.1 px. `findChessboardCornersSB` (OpenCV ≥ 4.3)
is a newer, more robust detector with sub-pixel accuracy built in.

In [ ]:
gray = cv2.imread(used[0], cv2.IMREAD_GRAYSCALE)
ret, c0 = cv2.findChessboardCorners(gray, (nx, ny), None)
c0 = c0.reshape(-1, 1, 2)
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
c1 = cv2.cornerSubPix(gray, c0.copy(), (11, 11), (-1, -1), crit).reshape(-1, 1, 2)
ok, c2 = cv2.findChessboardCornersSB(gray, (nx, ny), cv2.CALIB_CB_EXHAUSTIVE + cv2.CALIB_CB_ACCURACY)
print("mean shift by cornerSubPix:        %.3f px" % np.linalg.norm(c1 - c0, axis=-1).mean())
print("mean difference subpix vs SB:      %.3f px" % np.linalg.norm(c1.reshape(-1, 2) - c2.reshape(-1, 2), axis=-1).mean())

k = 20; x, y = c1[k, 0]
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(gray[int(y) - 10:int(y) + 11, int(x) - 10:int(x) + 11], cmap="gray", extent=(int(x) - 10.5, int(x) + 10.5, int(y) + 10.5, int(y) - 10.5))
ax.plot(*c0[k, 0], "r+", ms=25, mew=3, label="findChessboardCorners")
ax.plot(*c1[k, 0], "g+", ms=25, mew=3, label="cornerSubPix")
ax.plot(*c2.reshape(-1, 2)[k], "c+", ms=25, mew=3, label="SB detector"); ax.legend(); plt.show()

## Your turn

1. Time `findChessboardCorners` with and without the flag `cv2.CALIB_CB_FAST_CHECK` on an image **without** a board
   (e.g. a black image). Why is that flag useful in a live camera loop?
2. Run `findChessboardCornersSB` on the image that failed in section 5. Does it succeed? Why (not)?
3. Save `objpoints`, `imgpoints` and the image size with `pickle` (like Udacity's `wide_dist_pickle.p`) so Lab 6C
   could load them.